In [1]:
#Chat workflow

In [2]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import Annotated, TypedDict
from pydantic import BaseModel, Field
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from dotenv import load_dotenv

In [3]:
load_dotenv()  # Load environment variables from .env file

True

In [4]:
model= ChatOpenAI(model_name="gpt-4", temperature=0.7, max_tokens=1000)

In [5]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages] 
    ##Here we are using reducer concept.
    #BaseMessage is a class that represents a message in the chat. It can be either a HumanMessage, AIMessage, or SystemMessage. The add_messages function is a reducer that adds messages to the state.

In [6]:
def chat_node(state: ChatState):
    message = state["messages"]
    response = model.invoke(message)
    return {"messages": [response]}

In [8]:
graph= StateGraph(ChatState)
#Add Nodes
graph.add_node('Chat_Node', chat_node)
#Add Edges
graph.add_edge(START, 'Chat_Node')
graph.add_edge('Chat_Node', END)
chatbot= graph.compile()

In [10]:
initial_state = {"messages": [HumanMessage(content="Hello, how are you?")]}
interact_messages= chatbot.invoke(initial_state)

In [15]:
interact_messages['messages'][-1].content

"As an artificial intelligence, I don't have feelings, but I'm here and ready to assist you. How can I help you today?"

In [16]:
while True:
    user_input= input("User: ")
    print("User: ", user_input)
    if user_input.lower() in ['exit', 'quit']:
        print("Exiting the chat.")
        break
    intial_state = {"messages": [HumanMessage(content=user_input)]}
    interact_messages= chatbot.invoke(intial_state)
    print("Bot: ", interact_messages['messages'][-1].content)

User:  Remember me?
Bot:  As an AI, I don't have the ability to remember individuals or past interactions. I'm designed to respect user privacy and confidentiality.
User:  Okay np
Bot:  Great! If you have any other questions or need further assistance, feel free to ask.
User:  What I asked you before?
Bot:  As an AI, I don't have the ability to recall previous interactions or questions. Can you please specify or ask again?
User:  Okay let's discuss someting interesting topic on AI
Bot:  Sure, how about discussing the use of AI in healthcare? 

Artificial intelligence is transforming the healthcare industry in various ways. It's being used to predict diseases, improve diagnosis, and personalize treatments. 

One of the most significant uses of AI in healthcare is in predictive analytics. AI algorithms can use data from medical records to predict disease outcomes or the likelihood of a patient getting a certain disease. This can help doctors to intervene earlier and potentially save live